In [1]:
import yaml
import graphlib

In [5]:
with open("../conf/config.yaml") as f:
    config = yaml.safe_load(f)

config

{'estimators': {'model_A': {'model': 'mock_transformer',
   'datamodule': 'padded',
   'dataset': 'fair_universe',
   'trainer': 'default',
   'expands': {'systematics': {'up_qcd': {'dataset': 'fair_universe_local',
      'dataset_override': {'labels_columns': ['PRI_n_jets']}}},
    'ensembles': {'num_ensembles': 2, 'aggregation_method': 'mean'},
    'folds': 2}},
  'model_B': {'model': 'mock_transformer',
   'datamodule': 'padded',
   'dataset': 'fair_universe',
   'trainer': 'default',
   'requires': ['model_A'],
   'expands': {'ensembles': {'num_ensembles': 1,
     'aggregation_method': 'max'}}}}}

In [7]:
graph = {
    name: set(cfg.get("requires", []))
    for name, cfg in config["estimators"].items()
}
graph

{'model_A': set(), 'model_B': {'model_A'}}

In [30]:
config = {
    "estimators": {
        "model_A": {
            "model": "mock_transformer",
            "datamodule": "padded",
            "dataset": "fair_universe",
            "trainer": "default",
            "expands": {
                "systematics": {
                    "up_qcd": {"dataset": "fair_universe_local", "dataset_override": {"labels_columns": ["PRI_n_jets"]}}
                },
                "ensembles": {"num_ensembles": 2, "aggregation_method": "mean"},
                "folds": 2,
            },
        },
        "model_B": {
            "model": "mock_transformer",
            "datamodule": "padded",
            "dataset": "fair_universe",
            "trainer": "default",
            "requires": ["model_A"],
            "expands": {"ensembles": {"num_ensembles": 1, "aggregation_method": "max"}},
        },
    }
}

In [31]:
def validate_graph(config):
    estimators = set(config["estimators"])

    graph = {}
    for name, est in config["estimators"].items():
        deps = set(est.get("requires", []))

        missing = deps - estimators
        if missing:
            raise ValueError(f"{name} depends on undefined estimators {missing}")

        graph[name] = deps
    
    ts = graphlib.TopologicalSorter(graph)
    list(ts.static_order())

validate_graph(config)